## Install Required Dependecies

In [ ]:
!pip install langchain_openai

In [ ]:
!pip install -qU langchain-community faiss-cpu

In [ ]:
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.4 MB/s eta 0:00:00


## Necessary Imports

In [ ]:
# Necessary Imports

import os
import csv
import pandas as pd
from openai import OpenAI
from google.colab import userdata

from enum import Enum
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate


from abc import ABC, abstractmethod
from typing import Any, List, Tuple, Iterator, Dict

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings


from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain_core.documents import Document

from tqdm import tqdm

from langchain_groq import ChatGroq

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load OPENAI API KEY

api_key = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = api_key

In [ ]:
# Load GROQ API KEY

groq_api_key = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = groq_api_key

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

## Intent Classification Layer

In [ ]:
class IntentCategory(str, Enum):
    FACTUAL_FAQ = "FACTUAL_FAQ"
    PROCEDURAL_GUIDANCE = "PROCEDURAL_GUIDANCE"
    COMMUNITY_EXPERIENCE = "COMMUNITY_EXPERIENCE"
    EVIDENCE_BASED = "EVIDENCE_BASED"
    COMPLEX_MULTI_HOP = "COMPLEX_MULTI_HOP"


class IntentOutput(BaseModel):
    intent: IntentCategory = Field(
        description="The single most appropriate intent category for the user query"
    )


INTENT_CLASSIFIER_SYSTEM_PROMPT = """
You are an intent classification engine for a COVID-19 question-answering system.

Your task:
Given a user query, classify it into EXACTLY ONE intent category.

You must always return one category, even if the query does not clearly match any rules.
Use semantic understanding, not just keywords.

--------------------
INTENT CATEGORIES
--------------------

FACTUAL_FAQ:
- Short, direct factual questions
- Definitions, symptoms, transmission, vaccines, timelines
- Example: "What are the symptoms of COVID-19?"

PROCEDURAL_GUIDANCE:
- Questions asking for steps, actions, prevention, or advice
- Often includes "how", "what should I do", "can I", but do NOT rely only on keywords
- Example: "How can I protect myself from COVID-19?"

COMMUNITY_EXPERIENCE:
- Informal, opinionated, rumor-based, or social media influenced queries
- Mentions of people, beliefs, claims, or uncertainty
- Example: "People say masks don't work, is that true?"

EVIDENCE_BASED:
- Research-oriented, scientific, or evidence-seeking questions
- Mentions studies, data, research, clinical findings, or comparisons
- Example: "What does research say about COVID survival on surfaces?"

COMPLEX_MULTI_HOP:
- Long, multi-part, ambiguous, or mixed-intent questions
- Requires synthesising multiple facts or evidence
- Use this ONLY when no single intent clearly dominates
- Example: "How does COVID affect elderly people and what precautions should families take?"

--------------------
DECISION RULES (GUIDANCE, NOT HARD RULES)
--------------------

1. Use semantic meaning first, keywords second.
2. If multiple intents seem present:
   - Choose the MOST dominant intent.
3. If the query is long, compound, or unclear:
   - Prefer COMPLEX_MULTI_HOP.
4. If safety or scientific grounding is implied:
   - Prefer EVIDENCE_BASED over FACTUAL_FAQ.
5. Never return multiple categories.
6. Never invent new categories.
7. Never explain your reasoning.

Return ONLY the intent category in structured form.
"""

intent_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

intent_classifier = intent_llm.with_structured_output(IntentOutput)

def classify_intent_llm(query: str) -> IntentCategory:
    result = intent_classifier.invoke(
        [
            {"role": "system", "content": INTENT_CLASSIFIER_SYSTEM_PROMPT},
            {"role": "user", "content": query}
        ]
    )
    return result.intent.value

## Query Expansion Layer

In [ ]:
class QueryExpander:
    """
    LLM-based query expansion for multi-query retrieval.
    """

    DELIMITER = "#next-question#"

    def __init__(
        self,
        model_name: str = "gpt-4o-mini",
        n_variations: int = 3,
        temperature: float = 0.3,
    ):
        self.n_variations = n_variations
        self.llm = ChatOpenAI(
            model=model_name,
            temperature=temperature,
        )

        self.prompt = PromptTemplate(
            input_variables=["query", "n", "delimiter"],
            template="""
You are an expert search query rewriter for retrieval-augmented generation systems.

Rewrite the following user query into {n} alternative versions.
Each version should reflect a different perspective or phrasing
while preserving the original intent.

Separate each rewritten query using the token:
{delimiter}

User query:
"{query}"
""",
        )

    def expand(self, query: str) -> List[str]:
        formatted_prompt = self.prompt.format(
            query=query,
            n=self.n_variations,
            delimiter=self.DELIMITER,
        )

        response = self.llm.invoke(formatted_prompt).content

        expanded_queries = [
            q.strip()
            for q in response.split(self.DELIMITER)
            if q.strip()
        ]

        # Always include original query
        return [query] + expanded_queries


## Multi FAISS Retriever Function

In [ ]:
class MultiFAISSRetriever:
    """
    Concurrent retriever supporting:
    - multiple FAISS indexes
    - multiple expanded queries
    - optional similarity scores
    """

    def __init__(
        self,
        vectorstores: List[FAISS],
        top_k: int = 5,
        max_workers: int = 8,
        use_scores: bool = False,
    ):
        self.vectorstores = vectorstores
        self.top_k = top_k
        self.max_workers = max_workers
        self.use_scores = use_scores

    def _search(
        self, vs: FAISS, query: str
    ) -> List[Tuple[Document, float]] | List[Document]:
        """
        Internal helper to perform either:
        - similarity_search
        - similarity_search_with_score
        """
        if self.use_scores:
            return vs.similarity_search_with_score(query, k=self.top_k)
        return vs.similarity_search(query, k=self.top_k)

    def retrieve(
        self, queries: List[str]
    ) -> List[Document] | List[Tuple[Document, float]]:
        """
        Executes concurrent retrieval across all queries and indexes.
        Deduplicates documents across sources.
        """

        seen_doc_ids = set()
        results = []

        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = [
                executor.submit(self._search, vs, q)
                for vs in self.vectorstores
                for q in queries
            ]

            for future in as_completed(futures):
                retrieved = future.result()

                for item in retrieved:
                    if self.use_scores:
                        doc, score = item
                    else:
                        doc = item
                        score = None

                    # Robust deduplication key
                    doc_id = (
                        doc.metadata.get("id")
                        if doc.metadata and "id" in doc.metadata
                        else hash(doc.page_content)
                    )

                    if doc_id in seen_doc_ids:
                        continue

                    seen_doc_ids.add(doc_id)

                    if self.use_scores:
                        results.append((doc, score))
                    else:
                        results.append(doc)

        return results


## Context Retrieval Layer

In [ ]:
# Load Multiple FAISS Vector Stores from Google Drive

def load_faiss_vectorstores(
    vectorstore_paths: List[str],
    embedding_model: OpenAIEmbeddings,
) -> List[FAISS]:
    """
    Loads multiple FAISS vector stores from disk.
    """
    vectorstores = []

    for path in vectorstore_paths:
        vs = FAISS.load_local(
            folder_path=path,
            embeddings=embedding_model,
            allow_dangerous_deserialization=True,  # required for FAISS
        )
        vectorstores.append(vs)

    return vectorstores

# Vector Stores Path

vectorstore_paths = {
  "source_1": "/content/drive/MyDrive/MS-LJMU/Vector-Store/Store-1-3072-Kaggle-COVID-19-FAQ",

  "source_2": "/content/drive/MyDrive/MS-LJMU/Vector-Store/Store-2-3072-COVID-QA-community",

  "source_3": "/content/drive/MyDrive/MS-LJMU/Vector-Store/Store-3-3072-COUGH-FAQ-ENG",

  "source_4": "/content/drive/MyDrive/MS-LJMU/Vector-Store/Store-4-3072-COVID-QA-MASTER"

}

# Intent Routing

INTENT_ROUTING = {
    "FACTUAL_FAQ": ["source_1", "source_3"],
    "PROCEDURAL_GUIDANCE": ["source_1", "source_3"],
    "COMMUNITY_EXPERIENCE": ["source_2"],
    "EVIDENCE_BASED": ["source_4"],
    "COMPLEX_MULTI_HOP": ["source_1", "source_3", "source_4"],
}

## Post-Retrieval Layer

### Reranking (Cross-Encoders)

The most prominent post-retrieval technique discussed is reranking, which addresses the limitations of standard vector searches.

- The Problem: Initial searches (bi-encoders) rely on distance-based similarity between query and document embeddings, which may occasionally retrieve documents that are semantically close but contextually unaligned with the user's true intent.

- The Solution: A cross-encoder model is used to score each retrieved chunk against the original user query. Unlike bi-encoders, cross-encoders can identify more complex and nuanced relationships between the question and the text.

- The Workflow: The system retrieves a broader pool of candidates—typically N×K chunks if query expansion was used—and then the reranker reorders them, allowing the system to keep only the absolute best top K results for the final prompt.

In [ ]:
from sentence_transformers import CrossEncoder

In [ ]:
class CrossEncoderReranker:
    """
    Cross-encoder reranker for post-retrieval optimisation.
    """

    def __init__(
        self,
        model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
        top_k: int = 5,
        batch_size: int = 16,
    ):
        self.model = CrossEncoder(model_name)
        self.top_k = top_k
        self.batch_size = batch_size

    def rerank(
        self,
        query: str,
        documents: List[Document],
    ) -> List[Tuple[Document, float]]:
        """
        Reranks retrieved documents using cross-encoder scores.
        """

        pairs = [(query, doc.page_content) for doc in documents]

        scores = self.model.predict(
            pairs,
            batch_size=self.batch_size,
        )

        reranked = list(zip(documents, scores))
        reranked.sort(key=lambda x: x[1], reverse=True)

        return reranked[: self.top_k]


### Prompt Compression (Context Refinement Layer)

Why Prompt Compression?

Even after reranking:
- Chunks may be verbose
- Redundant details increase cost
- Long prompts → attention bias + hallucination

Goal:
- Preserve meaning, remove noise.

In [ ]:
class PromptCompressor:
    """
    Compresses retrieved documents while preserving factual content.
    """

    def __init__(
        self,
        model_name: str = "gpt-4o-mini",
        temperature: float = 0.0,
        max_tokens: int = 256,
    ):
        self.llm = ChatOpenAI(
            model=model_name,
            temperature=temperature,
        )

        self.prompt = PromptTemplate(
            input_variables=["query", "context"],
            template="""
You are a system that compresses retrieved context for a question-answering system.

Given the user query and the retrieved text:
- Remove irrelevant or redundant information
- Preserve all medically important facts
- Do NOT add new information
- Keep the output concise and factual

User query:
{query}

Retrieved context:
{context}

Compressed context:
""",
        )

        self.max_tokens = max_tokens

    def compress(
        self,
        query: str,
        documents: List[Document],
    ) -> List[str]:
        compressed_chunks = []

        for doc in documents:
            formatted_prompt = self.prompt.format(
                query=query,
                context=doc.page_content,
            )

            response = self.llm.invoke(
                formatted_prompt,
                max_tokens=self.max_tokens,
            )

            compressed_chunks.append(response.content.strip())

        return compressed_chunks


### Post Retrieval Optimisation Pipeline

In [ ]:
def post_retrieval_optimisation(
    query: str,
    retrieved_docs_with_scores: List[Tuple[Document, float]],
    rerank_top_k: int = 5,
) -> List[str]:
    """
    Post-retrieval optimisation:
    1. Cross-encoder reranking
    2. Prompt compression
    """

    # Strip FAISS scores
    retrieved_docs = [doc for doc, _ in retrieved_docs_with_scores]

    # Reranking
    reranker = CrossEncoderReranker(top_k=rerank_top_k)
    reranked_docs_with_scores = reranker.rerank(
        query=query,
        documents=retrieved_docs,
    )

    reranked_docs = [doc for doc, _ in reranked_docs_with_scores]

    # Prompt Compression
    compressor = PromptCompressor()
    compressed_context = compressor.compress(
        query=query,
        documents=reranked_docs,
    )

    return compressed_context


## Generation Layer

### Prompt Engineering

In [ ]:
SYSTEM_PROMPT = """
You are a helpful, factual, and cautious AI assistant.
Your task is to answer user questions using ONLY the provided context.
If the answer cannot be found in the context, clearly say:
"I do not have enough information to answer this question."

Do not make assumptions.
Do not add external knowledge.
Be concise and accurate.
"""


INSTRUCTION_PROMPT = """
Context:
{context}

User Question:
{question}

Instructions:
- Use the context above as your primary source of information
- Do not hallucinate or fabricate facts
- If the context is insufficient, explicitly say so

Answer:
"""


def build_augmented_prompt(
    question: str,
    context_chunks: List[str],
) -> List[dict]:
    """
    Builds a chat-style prompt with system + user messages.
    """

    context_text = "\n\n".join(
        f"[Source {i+1}]\n{chunk}"
        for i, chunk in enumerate(context_chunks)
    )

    user_prompt = INSTRUCTION_PROMPT.format(
        context=context_text,
        question=question,
    )

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


### Inference Service

In [ ]:
class GenerationOutput(BaseModel):
    """
    Structured output schema for final RAG generation.
    """
    llm_response: str = Field(
        description="Final grounded answer generated using the provided context."
    )

In [ ]:
# Closed-Source

class StructuredInferenceService:
    """
    Final RAG inference service with enforced structured output.
    """

    def __init__(
        self,
        model_name: str,
        temperature: float = 0,
        max_tokens: int = 2024,
    ):
        self.llm = ChatOpenAI(
            model=model_name,
            temperature=temperature,
            max_tokens=max_tokens,
        ).with_structured_output(GenerationOutput)

    def generate(self, messages: List[Dict]) -> GenerationOutput:
        """
        Generates a structured response.
        """
        return self.llm.invoke(messages)


# Open-Source

class OpenSourceStructuredInferenceService:
    """
    Final RAG inference service with enforced structured output.
    """

    def __init__(
        self,
        model_name: str,
        temperature: float = 0,
        max_tokens: int = 2024,
    ):
        self.llm = ChatGroq(
            model=model_name,
            temperature=temperature,
            max_tokens=max_tokens,
        ).with_structured_output(GenerationOutput)

    def generate(self, messages: List[Dict]) -> GenerationOutput:
        """
        Generates a structured response.
        """
        return self.llm.invoke(messages)

### Pre to Post-Retrieval Pipelines

In [ ]:
def pre_to_postretrieval_pipeline(user_query: str):

  # Indentify the intent and load respective vector stores
  intent = classify_intent_llm(user_query)
  sources = INTENT_ROUTING[intent]
  stores_paths = [vectorstore_paths[source] for source in sources]

  indexes = load_faiss_vectorstores(
      vectorstore_paths=stores_paths,
      embedding_model=embeddings,
  )

  # Query Expansion
  expander = QueryExpander(n_variations=3)
  expanded_queries = expander.expand(user_query)

  # Retrieve documents (Multi Index, Multi Query Retrieval)
  retriever = MultiFAISSRetriever(
      vectorstores=indexes,
      top_k=5,
      use_scores=True
  )

  retrieved_docs_with_scores = retriever.retrieve(expanded_queries)

  # Post Retrieval Optimization Pipeline
  compressed_context = post_retrieval_optimisation(
      query=user_query,
      retrieved_docs_with_scores=retrieved_docs_with_scores,
      rerank_top_k=5
  )

  return compressed_context

### Main Inference Service

In [ ]:
def main(user_query: str,
         model_name: str,
         open_source: False):

  if open_source:

    os_inference_llm = OpenSourceStructuredInferenceService(model_name=model_name)

    os_context = pre_to_postretrieval_pipeline(user_query)

    os_result = os_inference_llm.generate(
        messages=build_augmented_prompt(
            user_query,
            os_context,
      )
  )

    os_final_answer = os_result.llm_response

    return os_final_answer


  else:

    cs_inference_llm = StructuredInferenceService(model_name=model_name)

    cs_context = pre_to_postretrieval_pipeline(user_query)

    cs_result = cs_inference_llm.generate(
        messages=build_augmented_prompt(
            user_query,
            cs_context,
      )
  )

  cs_final_answer = cs_result.llm_response

  return cs_final_answer

In [ ]:
# 5 Large Language Models (2 Open-Source & 3 Closed-Source)

# "gpt-5-mini-2025-08-07"
# "gpt-4o-mini"
# "gpt-5.2-2025-12-11"

# "llama-3.3-70b-versatile"
# "llama-3.1-8b-instant"

### Closed-Source Inference

In [ ]:
# Ignore Warnings

import warnings
warnings.filterwarnings("ignore")

#### gpt-5-mini-2025

In [ ]:
# Case Study 1

# model_name = "gpt-5-mini-2025-08-07"

# user_query = "What are the symptoms of COVID-19?"

# gt = """The most common symptoms of COVID-19 are fever, tiredness, and dry cough. Some patients may have aches and pains, nasal congestion, runny nose, sore throat or diarrhea. These symptoms are usually mild and begin gradually. Some people become infected but dont develop any symptoms and don't feel unwell. Most people (about 80%) recover from the disease without needing special treatment. Around 1 out of every 6 people who gets COVID-19 becomes seriously ill and develops difficulty breathing. Older people, and those with underlying medical problems like high blood pressure, heart problems or diabetes, are more likely to develop serious illness. People with fever, cough and difficulty breathing should seek medical attention."""

respone = main(
    user_query="What are the symptoms of COVID-19?",
    model_name="gpt-5-mini-2025-08-07",
    open_source=False
)

print("****************************************************************************")
print(f"Final Answer: {respone}")
print("****************************************************************************")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


****************************************************************************
Final Answer: Common symptoms reported across the sources include:

- Most commonly: fever, tiredness/fatigue, and cough (often dry)
- Respiratory symptoms: shortness of breath or difficulty breathing, sore throat, nasal congestion, runny nose
- General/body symptoms: body aches or muscle aches, chills, headache, loss of appetite
- Sensory symptoms: new loss of smell or taste
- Gastrointestinal symptoms: diarrhea, nausea, vomiting, abdominal pain/discomfort

Additional points from the sources:
- Symptoms are usually mild and often begin gradually; some infected people have no symptoms (are asymptomatic).
- Severe illness can include high fever, severe cough, and difficulty breathing. About 1 in 6 people may become seriously ill, and around 80% recover without special treatment (per the provided context).
- Individuals with fever, cough, and difficulty breathing should seek medical attention.
******************

In [ ]:
# Case Study 2

# model_name = "gpt-5-mini-2025-08-07"

# user_query = "What can I do to protect myself and prevent the spread of disease?"

# gt = """Protection measures for everyone Stay aware of the latest information on the COVID-19 outbreak, available on the WHO website and through your national and local public health authority. Many countries around the world have seen cases of COVID-19 and several have seen outbreaks. Authorities in China and some other countries have succeeded in slowing or stopping their outbreaks. However, the situation is unpredictable so check regularly for the latest news. You can reduce your chances of being infected or spreading COVID-19 by taking some simple precautions: Regularly and thoroughly clean your hands with an alcohol-based hand rub or wash them with soap and water. Why? Washing your hands with soap and water or using alcohol-based hand rub kills viruses that may be on your hands. Maintain at least 1 meter (3 feet) distance between yourself and anyone who is coughing or sneezing. Why? When someone coughs or sneezes, they spray small liquid droplets from their nose or mouth which may contain virus. If you are too close, you can breathe in the droplets, including the COVID-19 virus if the person coughing has the disease. Avoid touching eyes, nose and mouth. Why? Hands touch many surfaces and can pick up viruses. Once contaminated, hands can transfer the virus to your eyes, nose or mouth. From there, the virus can enter your body and can make you sick. Make sure you, and the people around you, follow good respiratory hygiene. This means covering your mouth and nose with your bent elbow or tissue when you cough or sneeze. Then dispose of the used tissue immediately. Why? Droplets spread virus. By following good respiratory hygiene, you protect the people around you from viruses such as cold, flu and COVID-19. Stay home if you feel unwell. If you have a fever, cough and difficulty breathing, seek medical attention and call in advance. Follow the directions of your local health authority. Why? National and local authorities will have the most up to date information on the situation in your area. Calling in advance will allow your health care provider to quickly direct you to the right health facility. This will also protect you and help prevent spread of viruses and other infections. Keep up to date on the latest COVID-19 hotspots (cities or local areas where COVID-19 is spreading widely). If possible, avoid traveling to places ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â€šÂ¬Ã…â€œ especially if you are an older person or have diabetes, heart or lung disease. Why? You have a higher chance of catching COVID-19 in one of these areas."""

respone = main(
    user_query="What can I do to protect myself and prevent the spread of disease?",
    model_name="gpt-5-mini-2025-08-07",
    open_source=False
)

print("****************************************************************************")
print(f"Final Answer: {respone}")
print("****************************************************************************")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


****************************************************************************
Final Answer: Based on the provided information, you can protect yourself and help prevent spread of disease by doing the following:

- Practice good hand hygiene: wash hands frequently with soap and water or use an alcohol-based hand rub.
- Follow respiratory hygiene: cover your mouth and nose with your elbow or a tissue when coughing or sneezing, and dispose of tissues immediately.
- Wear a mask in crowded places.
- Maintain physical distance from others, keeping at least about 1 meter (3 feet) from people who are coughing or sneezing.
- Avoid close contact with sick individuals.
- Avoid touching your eyes, nose, and mouth.
- Stay home if you feel unwell; seek medical attention if you have fever, cough, or difficulty breathing.
- Stay informed and follow guidance from reliable sources (e.g., WHO and local health authorities) and local public health guidelines.
- Avoid traveling to COVID-19 hotspots, especial

#### gpt-4o-mini

In [ ]:
# Case Study 1

# model_name = "gpt-4o-mini"

# user_query = "What are the symptoms of COVID-19?"

respone = main(
    user_query="What are the symptoms of COVID-19?",
    model_name="gpt-4o-mini",
    open_source=False
)

print("****************************************************************************")
print(f"Final Answer: {respone}")
print("****************************************************************************")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


****************************************************************************
Final Answer: The symptoms of COVID-19 include fever, tiredness, dry cough, aches and pains, nasal congestion, runny nose, sore throat, diarrhea, chills, trouble breathing, loss of smell or taste, vomiting, and nausea. Symptoms are usually mild and can begin gradually, with some individuals remaining asymptomatic.
****************************************************************************


In [ ]:
# Case Study 2

# model_name = "gpt-4o-mini"

# user_query = "What can I do to protect myself and prevent the spread of disease?"

respone = main(
    user_query="What can I do to protect myself and prevent the spread of disease?",
    model_name="gpt-4o-mini",
    open_source=False
)

print("****************************************************************************")
print(f"Final Answer: {respone}")
print("****************************************************************************")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


****************************************************************************
Final Answer: To protect yourself and prevent the spread of disease, you can take the following measures:

1. Practice good respiratory hygiene by covering your mouth and nose with your elbow or tissue when coughing or sneezing, and dispose of tissues immediately.
2. Stay home if you feel unwell, and seek medical attention if you have a fever, cough, or difficulty breathing.
3. Follow local health authority guidelines for the latest information and avoid traveling to COVID-19 hotspots, especially if you are older or have underlying health conditions.
4. Wash your hands frequently with soap and water or an alcohol-based hand rub.
5. Wear a mask in crowded places.
6. Maintain at least 1 meter (3 feet) distance from anyone who is coughing or sneezing.
7. Avoid close contact with sick individuals.
8. Avoid touching your eyes, nose, and mouth to prevent transferring viruses from your hands into your body.
*********

#### gpt-5.2-2025

In [ ]:
# Case Study 1

# model_name = "gpt-5.2-2025-12-11"

# user_query = "What are the symptoms of COVID-19?"

respone = main(
    user_query="What are the symptoms of COVID-19?",
    model_name="gpt-5.2-2025-12-11",
    open_source=False
)

print("****************************************************************************")
print(f"Final Answer: {respone}")
print("****************************************************************************")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


****************************************************************************
Final Answer: From the provided sources, COVID-19 symptoms can include:
- Fever (sometimes high fever)
- Tiredness/fatigue
- Dry cough/cough
- Shortness of breath/trouble or difficulty breathing (can indicate more serious illness)
- Body or muscle aches/aches and pains
- Headache
- Chills
- Sore throat
- Nasal congestion/congestion or runny/stuffy nose
- New loss of smell or taste
- Loss of appetite
- Gastrointestinal symptoms: diarrhea, nausea, vomiting, abdominal pain/discomfort

The context also notes that symptoms are often mild and may begin gradually, and some people may be infected without developing any symptoms.
****************************************************************************


In [ ]:
# Case Study 2

# model_name = "gpt-5.2-2025-12-11"

# user_query = "What can I do to protect myself and prevent the spread of disease?"

respone = main(
    user_query="What can I do to protect myself and prevent the spread of disease?",
    model_name="gpt-5.2-2025-12-11",
    open_source=False
)

print("****************************************************************************")
print(f"Final Answer: {respone}")
print("****************************************************************************")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


****************************************************************************
Final Answer: To protect yourself and prevent the spread of disease, the context recommends:

- Practice good hygiene: wash your hands regularly with soap and water or use an alcohol-based hand rub/hand sanitizer.
- Follow good respiratory hygiene: cover your mouth and nose with your elbow or a tissue when coughing or sneezing, and dispose of tissues immediately.
- Keep distance from people who are coughing or sneezing (at least 1 meter/3 feet) and avoid close contact with sick individuals.
- Avoid touching your eyes, nose, and mouth.
- Stay home if you feel unwell.
- Seek medical attention if you have a fever, cough, or difficulty breathing.
- Stay informed and follow guidance from the WHO and local health authorities (including avoiding travel to COVID-19 hotspots, especially if you are older or have underlying health conditions).
****************************************************************************


### Open-Source Inference

#### llama-3.3-70b

In [ ]:
# Case Study 1

# model_name = "llama-3.3-70b-versatile"

# user_query = "What are the symptoms of COVID-19?"

respone = main(
    user_query="What are the symptoms of COVID-19?",
    model_name="llama-3.3-70b-versatile",
    open_source=True
)

print("****************************************************************************")
print(f"Final Answer: {respone}")
print("****************************************************************************")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


****************************************************************************
Final Answer: The symptoms of COVID-19 include fever, tiredness, dry cough, aches and pains, nasal congestion, runny nose, sore throat, diarrhea, chills, headache, loss of smell or taste, vomiting, nausea, abdominal pain/discomfort, and shortness of breath. Some infected individuals may remain asymptomatic or have mild symptoms, while others may develop serious illness, including difficulty breathing.
****************************************************************************


In [ ]:
# Case Study 2

# model_name = "llama-3.3-70b-versatile"

# user_query = "What can I do to protect myself and prevent the spread of disease?"

respone = main(
    user_query="What can I do to protect myself and prevent the spread of disease?",
    model_name="llama-3.3-70b-versatile",
    open_source=True
)

print("****************************************************************************")
print(f"Final Answer: {respone}")
print("****************************************************************************")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


****************************************************************************
Final Answer: To protect yourself and prevent the spread of disease, follow good respiratory hygiene by covering your mouth and nose with your elbow or tissue when coughing or sneezing, and dispose of tissues immediately. Stay home if you feel unwell, and seek medical attention if you have a fever, cough, or difficulty breathing. Practice good hygiene, such as regular handwashing, and follow public health guidelines. Maintain at least 1 meter (3 feet) distance from anyone who is coughing or sneezing. Stay informed about COVID-19 through reliable sources, clean your hands regularly with soap and water or an alcohol-based hand rub, and avoid touching your eyes, nose, and mouth.
****************************************************************************


#### llama-3.1-8b

In [ ]:
# Case Study 1

# model_name = "llama-3.1-8b-instant"

# user_query = "What are the symptoms of COVID-19?"

respone = main(
    user_query="What are the symptoms of COVID-19?",
    model_name="llama-3.1-8b-instant",
    open_source=True
)

print("****************************************************************************")
print(f"Final Answer: {respone}")
print("****************************************************************************")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


****************************************************************************
Final Answer: The most common symptoms of COVID-19 are fever, tiredness, and dry cough. Other symptoms may include aches and pains, nasal congestion, runny nose, sore throat, or diarrhea. Some infected individuals may remain asymptomatic. In severe cases, symptoms may include difficulty breathing, especially in older adults and those with underlying health conditions.
****************************************************************************


In [ ]:
# Case Study 2

# model_name = "llama-3.1-8b-instant"

# user_query = "What can I do to protect myself and prevent the spread of disease?"

respone = main(
    user_query="What can I do to protect myself and prevent the spread of disease?",
    model_name="llama-3.1-8b-instant",
    open_source=True
)

print("****************************************************************************")
print(f"Final Answer: {respone}")
print("****************************************************************************")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


****************************************************************************
Final Answer: To protect yourself and prevent the spread of disease, follow good respiratory hygiene by covering your mouth and nose with your elbow or tissue when coughing or sneezing, and dispose of tissues immediately. Stay home if you feel unwell, and seek medical attention if you have a fever, cough, or difficulty breathing. Follow local health authority guidelines for the latest information and avoid traveling to COVID-19 hotspots, especially if you are older or have underlying health conditions. Additionally, practice good hygiene, such as regular handwashing, wash your hands frequently, wear a mask in crowded places, maintain physical distance from others, stay informed about COVID-19, clean your hands regularly, maintain a safe distance from sick individuals, and avoid touching your eyes, nose, and mouth.
****************************************************************************
